# **Statistical Learning on CASdatasets-$\texttt{fremotorclaim}$**

## **Modeling Claim Count $N$**
In the following `code` cells, we compress the data frame, leaving only the explanatory risk factors and total claim counts for each risk profile. We then fit the Poisson GLM with and without `year` as a risk factor. The **likelihood-ratio test** finds that `year` is a statistically significant risk factor ($\Lambda = 746.07$, $q=2$, $p \approx 9.830848\times 10^{-163}$). 

Further, we compute the ratio of expected claim counts across years holding other risk factors fixed. With `year = 7` as the baseline, years 8 and 9 have expected claim counts approximately 1.09 and 1.17 times that of year 7, respectively.

Finally, we split the data frame into training (years 7–8) and testing (year 9) datasets, with `year` encoded as numeric so the fitted  trend can be extrapolated to `year = 9`.

In [1]:
load("euMTPL.rda")
df_eu = euMTPL 
df_eu$year = factor(df_eu$year)
df_eu$num_fcd = df_eu$num_fcg
df_eu$cost_fcd = df_eu$cost_fcg  
df_eu$num = df_eu$num_fcd + df_eu$num_cd 
df_eu$cost = df_eu$cost_fcd + df_eu$cost_cd 
df_eu = df_eu[, c(1,3,4,5,6,8,9,10,11,22,23)]

In [2]:
dim(df_eu); head(df_eu); summary(df_eu)

[1] 2373197      11

,policy_id,fuel_type,year,vehicle_category,vehicle_use,horsepower,gender,age,exposure,num,cost
,<int>,<fct>,<fct>,<fct>,<fct>,<int>,<fct>,<int>,<dbl>,<int>,<dbl>
1,1,B,7,1,1,14,M,77,0.48767123,0,0
2,2,B,7,1,1,12,M,40,0.01917808,0,0
3,4,B,7,1,1,14,M,75,0.03287671,0,0
4,5,B,7,1,1,13,M,48,0.04383562,0,0
5,6,B,7,1,1,12,F,54,0.04657534,0,0
6,8,B,7,1,1,12,F,34,0.07671233,0,0


   policy_id         fuel_type       year       vehicle_category vehicle_use 
 Min.   :      1   B      :1382612   7:788932   1:2360481        0 :  12670  
 1st Qu.: 641820   D      : 531255   8:789367   8:  12716        1 :2359462  
 Median :1280437   G      : 383832   9:794898                    4 :    145  
 Mean   :1290070   T      :  29637                               5 :    518  
 3rd Qu.:1939574   P      :  19289                               26:    402  
 Max.   :2595214   S      :  15131                                           
                   (Other):  11441                                           
   horsepower     gender           age            exposure     
 Min.   :  0.00   F: 893348   Min.   : 18.00   Min.   :0.0010  
 1st Qu.: 14.00   M:1479849   1st Qu.: 37.00   1st Qu.:0.3005  
 Median : 16.00               Median : 46.00   Median :0.6822  
 Mean   : 16.52               Mean   : 48.17   Mean   :0.6266  
 3rd Qu.: 19.00               3rd Qu.: 59.00   3rd Qu.:1

In [ ]:
# poisson.fit.full = glm(num ~ fuel_type + year + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_eu)
# poisson.fit.reduced = glm(num ~ fuel_type + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_eu)

In [ ]:
# saveRDS(poisson.fit.full, "fitted.model.full.rds")
# saveRDS(poisson.fit.reduced, "fit.model.reduced.rds")
# file.exists("fitted.model.full.rds")
# file.exists("fit.model.reduced.rds")

[1] TRUE

[1] TRUE

In [5]:
fit.full = readRDS("fitted.model.full.rds")
fit.reduced = readRDS("fit.model.reduced.rds")

In [8]:
anova(fit.reduced, fit.full, test = "LRT") 

,Resid. Df,Resid. Dev,Df,Deviance,Pr(>Chi)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2373181,951495.1,NA,NA,NA
2,2373179,950749.0,2,746.0717,9.830848e-163


In [7]:
coefs = summary(fit.full)$coefficients
year_coefs = coefs[grep("year", rownames(coefs)), ]
contrasts(df_eu$year); exp(year_coefs[, "Estimate"]) #contrasts() shows that year 7 is the baseline.

,8,9
7,0,0
8,1,0
9,0,1


year8    year9 
1.094518 1.169947

In [9]:
df_train = df_eu[df_eu$year %in% c("7", "8"), ]
df_test = df_eu[df_eu$year == "9", ]

df_train$year = as.numeric(as.character(df_train$year))
df_test$year = as.numeric(as.character(df_test$year))

fit.count.train = glm(num ~ fuel_type + year + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_train)
pred.count.test = predict(fit.count.train, newdata = df_test, type = "response")

### **Model Significance & Goodness-of-Fit Assessment**
The following `code` provide the results of **Likelihood-Ratio Test**, **Deviance Goodness-of-Fit Test** and **Pearson's Chi-Squared Test** respectively.

#### **Overall Model Significance (Likelihood-Ratio Test)**
$p$-value being essentialy $0$ shows that our risk factors actually have predictive power.

In [10]:
dif.dev = fit.count.train$null.deviance - fit.count.train$deviance
dif.df = fit.count.train$df.null - fit.count.train$df.residual
p_val_lrt = pchisq(dif.dev, df = dif.df, lower.tail = FALSE)
dif.dev; dif.df; p_val_lrt

[1] 4148.905

[1] 16

[1] 0

#### **Model Specification (Deviance Goodness-of-Fit Test)**
$p$-value being approximately 1 shows that Poisson GLM is well specified.

In [11]:
dev = fit.count.train$deviance
df_res = fit.count.train$df.residual
p_val_gof = pchisq(dev, df = df_res, lower.tail = FALSE)
dev; p_val_gof

[1] 1052743

[1] 1

#### **Overdispersion (Pearson's Chi-Squared Test)**
The large value of X2 shows that sum of squared standardised errors is large, and the estimate of the dispersion parameter (`est.phi = 2.14`) is the evidence of dispersion. 

In [12]:
fit.val = predict(fit.count.train, newdata = df_train, type = 'response')
X2 = sum((fit.val-df_train$num_cl)^2/fit.val) #sum of squared standardised errors
df_train_residual = fit.count.train$df.residual
p_val_X2 = pchisq(X2, df = df_train_residual, lower.tail = FALSE)
est.phi = X2/df_train_residual #estimate of dispersion parameter
X2; p_val_X2; est.phi

[1] 3376109

[1] 0

[1] 2.139104

## **Modeling Claim Severity $Y_j$**
In the following `code` cells we first remove individuals with 0 claim. Then, we compute the average claim amounts for each individual. 

In [22]:
df_sev = df_eu[df_eu$num_cl > 0, ] # remove insureds with no claim
df_sev$sev = (df_sev$cost_nc + df_sev$cost_cg + df_sev$cost_fcg + df_sev$cost_cd)/df_sev$num_cl #average claim amount
df_sev = df_sev[, c(1,2,3,4,5,6,7,8,10,15)]
df_sev

,policy_id,fuel_type,year,vehicle_category,vehicle_use,horsepower,gender,age,num_cl,sev
,<int>,<fct>,<fct>,<fct>,<fct>,<int>,<fct>,<int>,<int>,<dbl>
41,44,B,7,1,1,12,F,31,2,1692.4700
173,182,B,7,1,1,18,M,45,1,1800.0000
175,184,D,7,1,1,18,M,61,1,1632.0000
187,196,B,7,1,1,18,F,74,1,1800.0000
188,197,B,7,1,1,14,M,37,1,303.5900
194,203,B,7,1,1,15,M,53,1,6231.2300
206,215,D,7,1,1,19,F,60,2,1030.2450
382,396,B,7,1,1,15,M,60,2,1467.6800
446,463,B,7,1,1,12,M,29,1,2300.0000
